In [1]:
from datasets import load_dataset, Dataset
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statistics
from transformers import AutoTokenizer
from collections import Counter

## Contextualization
---

Unlike traditional methods (like TF-IDF or Word2Vec) which require aggressive cleaning to reduce vocabulary size, encoder architectures generally perform best with minimal intervention on the raw text.

- Stemming and Lemmatization: Encoder models often use Byte-Pair Encoding (BPE) for tokenization, which breaks words into sub-word units (e.g., 'running' might become 'run' and '##ing'). This sub-word tokenization implicitly handles morphological variations, automatically grouping inflections of the same root word. Forcing stemming or lemmatization before tokenization can inadvertently remove necessary linguistic information, often leading to reduced performance.

- Stop Word Management: It is generally not necessary to remove high-frequency words (stopwords). Transformer models, which are built on the attention principle, automatically learn to focus only on the words that impact the classification output, effectively downweighting common words. Removing stopwords can be problematic in cases like sentiment analysis where the absence of words like "not" or "too" changes the context.

- Case Handling: Case normalization (lowercasing) is typically managed by selecting the appropriate pre-trained model. If you choose an uncased model (e.g., bert-base-uncased), the model will internally convert all input text to lowercase. If you choose a cased model (e.g., bert-base-cased), capitalization is preserved, which can be important for tasks where proper nouns are critical. You do not need to manually perform case folding.

## Data Loading
---

In [ ]:
"""
HuggingFace Load
"""
# dataset = load_dataset("nick007x/arxiv-papers")
# Take first 100k examples from the train split
# dataset = dataset['train'].select(range(100_000))


"""
LOCAL LOAD
"""
# Path to parquet file
PARQUET_FILE_PATH = "./data/arxiv_papers/train.parquet"
# Load the parquet file
df = pd.read_parquet(PARQUET_FILE_PATH)
dataset_complete = Dataset.from_pandas(df, preserve_index=False)
dataset = dataset_complete.select(range(100_000))


print(dataset)

ValueError: Column 'train' doesn't exist.

## Data Preprocessing
---

This code filters the dataset to retain only the primary subjects with a sufficient number of samples. It first counts how many papers belong to each primary_subject, then identifies the subjects that have at least 20,000 examples. Finally, it filters the dataset to include only papers whose primary subject is in this set of frequent labels. This ensures the resulting dataset is focused on well-represented categories, reducing class imbalance and providing enough data for meaningful analysis or model training.

In [ ]:
threshold = 20000

counts = Counter(dataset["primary_subject"])
valid_labels = {label for label, cnt in counts.items() if cnt >= threshold}
filtered_ds = dataset.filter(lambda x: x["primary_subject"] in valid_labels)

This code processes a dataset of abstracts by tokenizing each text using the Longformer tokenizer, collecting the lengths of the resulting token sequences, and analyzing their distribution. It computes key statistics such as the median, average, minimum, and maximum token lengths, while also visualizing the distribution with a histogram to identify patterns like long or short documents. This helps ensure the dataset is well-understood and suitable for model training, highlighting sequences that may be too short to provide meaningful context or too long to handle efficiently.

In [ ]:
def plot_histogram(data):
    data = np.array(data)
    over_200 = data[data > 200]
    print(len(over_200))

    sns.histplot(data, bins=20, kde=True)
    plt.xlabel('Token count')
    plt.ylabel('Density')
    plt.title('Distribution of Token Lengths')
    plt.show()

def tokenize_dataset(ds, tokenizer):
    def tokenize_fn(batch):
        return tokenizer(batch["abstract"], padding=False, truncation=False, return_length=True)

    tokenized_ds = ds.map(tokenize_fn, batched=True)
    lengths = [len(ids) for ids in tokenized_ds["input_ids"]]
    return tokenized_ds, lengths


def main(ds, tokenizer):
    tokenized_df, lengths = tokenize_dataset(ds, tokenizer)

    print(f"Median token length: {statistics.median(lengths)}")
    print(f"Average token length: {statistics.mean(lengths)}")
    print(f"Min token length: {min(lengths)}")
    print(f"Max token length: {max(lengths)}")
    print(f"Total samples: {len(lengths)}")

    plot_histogram(lengths)
    return tokenized_df, lengths

if __name__ == "__main__":
    tokenizer = AutoTokenizer.from_pretrained("allenai/longformer-base-4096")
    tokenized_df, lengths = main(filtered_ds, tokenizer)

The code filters the dataset to keep only tokenized texts whose lengths are between MIN_LEN and MAX_LEN tokens. This ensures that very short texts (too little context) and very long texts (memory-heavy) are removed, making the dataset more consistent and efficient for model training.

In [ ]:
MIN_LEN = 100
MAX_LEN = 500 # you can increase up to 4096 if needed

dataset = tokenized_df.filter(lambda x: MIN_LEN <= x["length"] <= MAX_LEN)
print(dataset.shape)

## Label Encoding
---

This code prepares the dataset for model training by converting categorical labels into numerical IDs. First, it extracts all unique `primary_subject` labels, sorts them, and creates two mappings: `label2id` to map each label to a unique integer, and `id2label` to reverse the mapping. Then, it defines a function `encode_labels` that adds a new `"label"` field to each dataset example, containing the integer ID corresponding to its `primary_subject`. Finally, it applies this function to the entire dataset using `.map()`, resulting in a dataset where each example has both its original text and a numerical label suitable for machine learning models.

In [ ]:
labels = list(set(dataset['primary_subject']))
labels.sort()
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

In [ ]:
def encode_labels(example):
    example["label"] = label2id[example["primary_subject"]]
    return example

dataset = dataset.map(encode_labels)

## Model Call
---

Just an simple example of how to call the model.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "allenai/longformer-base-4096",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)